# Power Analizi — Cerrahi Sınır Pozitif (Histerektomi + Re-LEEP) vs Cerrahi Sınır Negatif

**Nasıl çalıştırılır**

1. Çalışma zamanı türü **Python 3** olmalı (*Çalışma zamanı → Çalışma zamanı türünü değiştir*).
   R kodları `%%R` hücrelerinde çalışır. Colab'ın "R" çalışma zamanında Google Drive bağlanamadığı için bu yol kullanılıyor.
2. Hücreleri yukarıdan aşağıya sırayla çalıştırın (veya *Çalışma zamanı → Tümünü çalıştır*).
   İlk hücre Google Drive izni ister.
3. Paketler her yeni Colab oturumunda bir kez kurulur (yaklaşık 1–3 dk).
4. Çıktılar Drive'daki **Tez_Fatma_Beyza_Turk** klasörüne kaydedilir:
   `Power_Analizi_YYYY-AA-GG_v1` → aynı gün tekrar çalıştırılırsa `_v2`, `_v3` …; ertesi gün yeni tarih ve yine `_v1`.

Grup tanımı, saat dilimi gibi ayarlar **"3. Ayarlar, verileri okuma…"** hücresinin başındaki **AYARLAR** bölümündedir.
Excel dosyalarını güncelledikten sonra not defterini yeniden çalıştırmanız yeterli; hasta sayıları Excel'den otomatik alınır.

Çıktılar: `.docx` (Word tabloları), `.xlsx` (tüm sonuçlar ve kontrol tabloları), `.html` (üç tablo tek sayfada).

## 1. Google Drive + R

In [ ]:
# ======================================================================
# 1. GOOGLE DRIVE'I BAĞLA + R'Yİ ETKİNLEŞTİR
# ======================================================================
# Bu not defteri PYTHON çalışma zamanında açılır; R kodları alttaki
# %%R hücrelerinde çalışır. (Colab'ın "R" çalışma zamanında Google Drive
# bağlanamadığı için bu yol kullanılıyor.)
#
# Çalışma zamanı > Çalışma zamanı türünü değiştir > Python 3   olmalı.

from google.colab import drive
drive.mount('/content/drive')

import importlib.util, subprocess, sys
if importlib.util.find_spec("rpy2") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rpy2"], check=True)

%load_ext rpy2.ipython

## 2. R paketleri

In [ ]:
%%R
# ======================================================================
# 2. PAKETLER  (Colab'da her yeni oturumda bir kez kurulur, ~1-3 dk)
# ======================================================================
# Posit Package Manager'ın derlenmiş Linux paketleri kullanılır; böylece
# kurulum kaynak koddan derlemeye göre çok daha hızlı olur.

os_bilgi   <- tryCatch(readLines("/etc/os-release"), error = function(e) character(0))
ubuntu_kod <- sub("^VERSION_CODENAME=", "", grep("^VERSION_CODENAME=", os_bilgi, value = TRUE))
if (length(ubuntu_kod) == 0 || ubuntu_kod[1] == "") ubuntu_kod <- "jammy"

options(
  repos = c(CRAN = paste0("https://packagemanager.posit.co/cran/__linux__/", ubuntu_kod[1], "/latest")),
  HTTPUserAgent = sprintf(
    "R/%s R (%s)",
    getRversion(),
    paste(getRversion(), R.version["platform"], R.version["arch"], R.version["os"])
  ),
  Ncpus = max(1L, parallel::detectCores())
)

paketler <- c(
  "readxl", "pwr", "dplyr", "tidyr",
  "gt", "flextable", "officer", "writexl"
)

eksik_paketler <- paketler[!vapply(paketler, requireNamespace, logical(1), quietly = TRUE)]

if (length(eksik_paketler) > 0) {
  cat("Kuruluyor:", paste(eksik_paketler, collapse = ", "), "\n")
  install.packages(eksik_paketler)
}

hala_eksik <- paketler[!vapply(paketler, requireNamespace, logical(1), quietly = TRUE)]
if (length(hala_eksik) > 0) {
  stop("Kurulamayan paket(ler): ", paste(hala_eksik, collapse = ", "))
}

suppressPackageStartupMessages({
  library(readxl)
  library(pwr)
  library(dplyr)
  library(tidyr)
  library(gt)
  library(flextable)
  library(officer)
  library(writexl)
})

cat("Tüm paketler hazır. R sürümü:", R.version.string, "\n")

## 3. Ayarlar, verileri okuma ve grupları oluşturma

In [ ]:
%%R
# ======================================================================
#              POWER / ÖRNEKLEM BÜYÜKLÜĞÜ ANALİZİ
#   CERRAHİ SINIR POZİTİF (HİSTEREKTOMİ + RE-LEEP)  vs  CERRAHİ SINIR NEGATİF
# ======================================================================
# Alfa = 0.05 | Güç = 0.80 | İki yönlü test
# Sürekli: Cohen's d (0.20 - 0.80) | Kategorik: Cohen's h (0.20 - 0.80)
#
# Sabit grup = Cerrahi sınır POZİTİF hastalar
#              (histerektomi + re-LEEP dosyalarındaki benzersiz kişi sayısı,
#               Excel'den otomatik alınır)
# Karşı grup = Cerrahi sınır NEGATİF hastalar
#              (gerekli sayı hesaplanır; ayrıca eldeki negatif N ile güç gösterilir)
# ======================================================================


# ======================================================================
# 3. ORTAMI TEMİZLE
# ======================================================================

rm(list = ls())
graphics.off()


# ======================================================================
# 4. AYARLAR — GEREKİRSE YALNIZCA BU BÖLÜMÜ DEĞİŞTİRİN
# ======================================================================

ana_klasor <- "/content/drive/MyDrive/Tez_Fatma_Beyza_Turk"

dosya_histerektomi <- file.path(ana_klasor, "cerrahi_sinir_pozitif_histerektomi.xlsx")
dosya_releep       <- file.path(ana_klasor, "cerrahi_sinir_pozitif_releep.xlsx")
dosya_negatif      <- file.path(ana_klasor, "cerrahi_sinir_negatif.xlsx")

# Çıktılar aynı klasöre şu adla kaydedilir:
#   Power_Analizi_<bugünün tarihi>_v1  (aynı gün tekrar çalışırsa _v2, _v3 ...)
cikti_adi   <- "Power_Analizi"
saat_dilimi <- "America/Chicago"   # Türkiye saatiyle tarih için: "Europe/Istanbul"

# GRUP TANIMI (demografik analiz not defterindeki ayarla aynı olmalı)
#   "dosya" : pozitif dosyalardaki herkes POZİTİF, negatif dosyadaki herkes NEGATİF
#   "kod"   : Cerrahi_Sinir_Durumu sütununa göre
#             POZİTİF = histerektomi / re-LEEP dosyasında kodu pozitif_kodlar içinde olanlar
#             NEGATİF = negatif dosyada kodu negatif_kodlar içinde olanlar
#             (kodu boş olanlar analiz dışında kalır)
grup_tanimi    <- "dosya"
pozitif_kodlar <- c(1)   # 1 = Pozitif   (1 mm yakın sınırı da saymak için: c(1, 3))
negatif_kodlar <- c(0)   # 0 = Negatif

alfa                <- 0.05
hedef_power         <- 0.80
maksimum_karsi_grup <- 100000L
etki_degerleri      <- c(0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80)


# ======================================================================
# 5. KLASÖR VE DOSYALAR VAR MI?
# ======================================================================

if (!dir.exists(ana_klasor)) {
  stop(
    "Klasör bulunamadı: ", ana_klasor,
    "\nİlk hücrede Google Drive bağlandı mı? Klasör adı doğru mu?"
  )
}

for (f in c(dosya_histerektomi, dosya_releep, dosya_negatif)) {
  if (!file.exists(f)) stop("Dosya bulunamadı: ", f)
}


# ======================================================================
# 6. YARDIMCI FONKSİYONLAR
# ======================================================================

# Tarihli + versiyonlu dosya adı
#   Aynı gün ilk çalıştırma -> Power_Analizi_2026-10-11_v1
#   Aynı gün tekrar         -> Power_Analizi_2026-10-11_v2, _v3 ...
#   Ertesi gün              -> Power_Analizi_2026-10-12_v1
versiyonlu_ad <- function(klasor, temel_ad, tarih) {
  onek    <- paste0(temel_ad, "_", tarih, "_v")
  mevcut  <- list.files(klasor)
  mevcut  <- mevcut[startsWith(mevcut, onek)]
  kalan   <- substring(mevcut, nchar(onek) + 1)
  nolar   <- suppressWarnings(as.integer(sub("^([0-9]+).*$", "\\1", kalan)))
  nolar   <- nolar[!is.na(nolar)]
  sonraki <- if (length(nolar) == 0) 1L else max(nolar) + 1L
  paste0(onek, sonraki)
}

# Hasta numarasını metne çevirir (11-12 haneli numaralar bilimsel gösterime dönmesin)
hasta_no_temizle <- function(x) {
  if (is.numeric(x)) {
    sonuc <- rep(NA_character_, length(x))
    dolu  <- !is.na(x)
    sonuc[dolu] <- formatC(x[dolu], format = "f", digits = 0, big.mark = "")
    return(sonuc)
  }
  x <- trimws(as.character(x))
  sayi <- suppressWarnings(as.numeric(x))
  bilimsel <- !is.na(sayi) & grepl("[eE]", x)
  x[bilimsel] <- formatC(sayi[bilimsel], format = "f", digits = 0, big.mark = "")
  sub("\\.0+$", "", x)
}

# Boş / "NA" / ÖRNEK satırlarını ayıklar
gercek_hasta_mi <- function(x) {
  !is.na(x) & x != "" & toupper(x) != "NA" &
    !grepl("^(ÖRNEK|ORNEK)", x, ignore.case = TRUE)
}

# Metin olarak girilmiş sayıları sayıya çevirir ("NA " -> boş, "39y" -> 39)
sayisal_yap <- function(x) {
  if (is.numeric(x)) return(as.numeric(x))
  x <- trimws(as.character(x))
  x <- gsub(",", ".", x, fixed = TRUE)
  x[toupper(x) %in% c("", "NA", "N/A", "-", ".")] <- NA
  sonuc  <- suppressWarnings(as.numeric(x))
  kurtar <- is.na(sonuc) & !is.na(x) & grepl("^[0-9]+(\\.[0-9]+)?\\s*[[:alpha:]]+$", x)
  sonuc[kurtar] <- as.numeric(sub("^([0-9]+(\\.[0-9]+)?).*$", "\\1", x[kurtar]))
  sonuc
}

# Her değişkenin Excel başlığında aranacak ifade (sütun kaymasını yakalamak için)
sutun_desenleri <- c(
  hasta_no     = "no$",
  sinir_durumu = "^cerrahi_sinir_d"
)

# Excel'i okur, istenen sütunları numarasıyla alır ve başlıklarını kontrol eder
excel_oku <- function(yol, sayfa, atla, sutunlar, kaynak_adi) {
  ham <- suppressMessages(read_excel(path = yol, sheet = sayfa, skip = atla))
  liste <- lapply(names(sutunlar), function(degisken) {
    no <- sutunlar[[degisken]]
    if (no > ncol(ham)) {
      stop(basename(yol), ": '", degisken, "' için ", no, ". sütun yok (dosyada ",
           ncol(ham), " sütun var).")
    }
    baslik <- names(ham)[no]
    if (!grepl(sutun_desenleri[[degisken]], baslik, ignore.case = TRUE, perl = TRUE)) {
      stop(basename(yol), ": '", degisken, "' için ", no, ". sütun bekleniyordu ama başlığı '",
           baslik, "'.\nExcel'de sütun sırası değişmiş olabilir; bu dosyanın sütun numaralarını düzeltin.")
    }
    ham[[no]]
  })
  names(liste) <- names(sutunlar)
  as_tibble(liste) %>%
    mutate(
      hasta_no     = hasta_no_temizle(hasta_no),
      excel_satiri = seq_len(n()) + atla + 1L,
      kaynak       = kaynak_adi
    ) %>%
    filter(gercek_hasta_mi(hasta_no))
}

# Boş tabloyu Excel'e yazılabilir hale getirir
bos_ise_bilgi <- function(veri) {
  if (is.null(veri) || nrow(veri) == 0) return(data.frame(Bilgi = "Kayıt yok"))
  as.data.frame(veri)
}

yuzde <- function(x) ifelse(is.na(x), "—", sprintf("%.1f%%", x * 100))


# ======================================================================
# 7. EXCEL DOSYALARINI OKU
# ======================================================================
# Sütun numaraları: 1 = A sütunu, 2 = B sütunu ...
#   Histerektomi : başlıklar 2. satırda (skip = 1), cerrahi sınır = 21. sütun
#   Re-LEEP      : başlıklar 4. satırda (skip = 3), cerrahi sınır = 21. sütun
#   Negatif      : başlıklar 1. satırda (skip = 0), cerrahi sınır = 25. sütun

histerektomi <- excel_oku(
  yol = dosya_histerektomi, sayfa = "Sayfa1", atla = 1,
  sutunlar = c(hasta_no = 1, sinir_durumu = 21),
  kaynak_adi = "Histerektomi"
)

releep <- excel_oku(
  yol = dosya_releep, sayfa = "Veri Toplama Formu", atla = 3,
  sutunlar = c(hasta_no = 1, sinir_durumu = 21),
  kaynak_adi = "Re-LEEP"
)

negatif <- excel_oku(
  yol = dosya_negatif, sayfa = "Sayfa1", atla = 0,
  sutunlar = c(hasta_no = 1, sinir_durumu = 25),
  kaynak_adi = "Negatif"
)

tum_kayitlar <- bind_rows(
  lapply(
    list(histerektomi, releep, negatif),
    function(v) mutate(v, sinir_durumu = sayisal_yap(sinir_durumu))
  )
)


# ======================================================================
# 8. CERRAHİ SINIR KODLARI — DOSYALARA GÖRE KONTROL
# ======================================================================
# Pozitif dosyalarda kodu 0/2/3 olan, negatif dosyada kodu 1 olan hastalar
# burada görünür. "kod" grup tanımını seçmek isteyip istemediğinize
# bu tabloya bakarak karar verebilirsiniz.

kod_etiketleri <- c(
  "0" = "0 = Negatif",
  "1" = "1 = Pozitif",
  "2" = "2 = Değerlendirilemedi",
  "3" = "3 = 1 mm uzaklıkta"
)

sinir_kod_tablosu <- tum_kayitlar %>%
  mutate(
    kod = as.character(sinir_durumu),
    kod = ifelse(
      is.na(kod), "Boş",
      ifelse(kod %in% names(kod_etiketleri), kod_etiketleri[kod], kod)
    ),
    kaynak = factor(kaynak, levels = c("Histerektomi", "Re-LEEP", "Negatif"))
  ) %>%
  count(kaynak, kod) %>%
  arrange(kod) %>%
  pivot_wider(names_from = kod, values_from = n, values_fill = 0) %>%
  arrange(kaynak) %>%
  rename(Dosya = kaynak)

cat("\n============================================================\n")
cat("CERRAHİ SINIR DURUMU KODLARI (dosyalara göre)\n")
cat("============================================================\n\n")
print(as.data.frame(sinir_kod_tablosu), row.names = FALSE)


# ======================================================================
# 9. GRUPLARI OLUŞTUR
# ======================================================================

if (grup_tanimi == "dosya") {
  tum_kayitlar <- tum_kayitlar %>%
    mutate(grup = if_else(kaynak == "Negatif", "Cerrahi sınır negatif", "Cerrahi sınır pozitif"))
} else if (grup_tanimi == "kod") {
  tum_kayitlar <- tum_kayitlar %>%
    mutate(grup = case_when(
      kaynak != "Negatif" & sinir_durumu %in% pozitif_kodlar ~ "Cerrahi sınır pozitif",
      kaynak == "Negatif" & sinir_durumu %in% negatif_kodlar ~ "Cerrahi sınır negatif",
      TRUE ~ NA_character_
    ))
} else {
  stop("grup_tanimi yalnızca \"dosya\" veya \"kod\" olabilir.")
}

grup_disi_kalanlar <- tum_kayitlar %>% filter(is.na(grup))
analiz_kayitlari   <- tum_kayitlar %>% filter(!is.na(grup))


# ======================================================================
# 10. TEKRARLAYAN HASTALAR
# ======================================================================
# Aynı grupta birden fazla satırı olan hasta tek kişi sayılır (ilk satırı alınır).
# Örn. hem histerektomi hem re-LEEP dosyasında olan hasta pozitif grupta 1 kez sayılır.

tekrarlayan_kayitlar <- analiz_kayitlari %>%
  group_by(grup, hasta_no) %>%
  filter(n() > 1) %>%
  ungroup() %>%
  arrange(grup, hasta_no)

analiz <- analiz_kayitlari %>%
  distinct(grup, hasta_no, .keep_all = TRUE)

# Hem pozitif hem negatif grupta görünen hasta varsa iki gruptan da çıkarılır
iki_grupta_olanlar <- analiz %>%
  count(hasta_no) %>%
  filter(n > 1) %>%
  pull(hasta_no)

if (length(iki_grupta_olanlar) > 0) {
  cat("\nUYARI: Hem pozitif hem negatif grupta görünen", length(iki_grupta_olanlar),
      "hasta analiz dışı bırakıldı:\n")
  print(iki_grupta_olanlar)
  analiz <- analiz %>% filter(!hasta_no %in% iki_grupta_olanlar)
}

analiz$grup <- factor(
  analiz$grup,
  levels = c("Cerrahi sınır pozitif", "Cerrahi sınır negatif")
)


# ======================================================================
# 11. DİNAMİK N
# ======================================================================

n_sabit        <- sum(analiz$grup == "Cerrahi sınır pozitif")
n_karsi_mevcut <- sum(analiz$grup == "Cerrahi sınır negatif")

if (is.na(n_sabit) || n_sabit < 2) {
  stop("Geçerli cerrahi sınır pozitif hasta sayısı hesaplanamadı.")
}


# ======================================================================
# 12. VERİ SAYILARINI RAPORLA
# ======================================================================

dosya_ozeti <- tum_kayitlar %>%
  mutate(kaynak = factor(kaynak, levels = c("Histerektomi", "Re-LEEP", "Negatif"))) %>%
  group_by(Dosya = kaynak) %>%
  summarise(
    `Satır sayısı`    = n(),
    `Benzersiz hasta` = n_distinct(hasta_no),
    .groups = "drop"
  )

grup_dagilimi <- analiz %>%
  mutate(kaynak = factor(kaynak, levels = c("Histerektomi", "Re-LEEP", "Negatif"))) %>%
  count(Grup = grup, Dosya = kaynak, name = "Hasta sayısı")

cat("\n============================================================\n")
cat("EXCEL DOSYALARINDAN OTOMATİK HASTA SAYISI\n")
cat("============================================================\n\n")
print(as.data.frame(dosya_ozeti), row.names = FALSE)

cat("\nGrup tanımı:", grup_tanimi, "\n")
cat("Aynı grupta tekrarlayan hasta sayısı:", n_distinct(tekrarlayan_kayitlar$hasta_no),
    "(her biri 1 kez sayıldı)\n")
cat("Grup tanımı dışında kalan satır:", nrow(grup_disi_kalanlar), "\n\n")

print(as.data.frame(grup_dagilimi), row.names = FALSE)

cat("\nCERRAHİ SINIR POZİTİF (sabit grup) N =", n_sabit, "\n")
cat("CERRAHİ SINIR NEGATİF (eldeki)     N =", n_karsi_mevcut, "\n")

## 4. Power hesapları

In [ ]:
%%R
# ======================================================================
# 13. TEMEL POWER AYARLARI
# ======================================================================

cat("\n============================================================\n")
cat("POWER ANALİZİ AYARLARI\n")
cat("============================================================\n\n")
cat("Sabit grup N (cerrahi sınır pozitif) =", n_sabit, "\n")
cat("Eldeki karşı grup N (cerrahi sınır negatif) =", n_karsi_mevcut, "\n")
cat("Alfa =", alfa, "\n")
cat("Hedef power =", hedef_power, "\n")
cat("Test = iki yönlü\n\n")


# ======================================================================
# 14. ETKİ BÜYÜKLÜĞÜ ETİKETİ
# ======================================================================

etki_etiketi <- function(x) {
  if (x < 0.50) return("Küçük")
  if (x < 0.80) return("Orta")
  return("Büyük")
}


# ======================================================================
#                  A. SÜREKLİ DEĞİŞKENLER
# ======================================================================

# ======================================================================
# 15. SÜREKLİ DEĞİŞKEN POWER FONKSİYONU
# ======================================================================

power_surekli <- function(n_karsi, n_sabit, d, alfa) {
  sonuc <- pwr::pwr.t2n.test(
    n1 = n_sabit,
    n2 = n_karsi,
    d = d,
    sig.level = alfa,
    alternative = "two.sided"
  )
  sonuc$power
}


# ======================================================================
# 16. GEREKLİ KARŞI GRUP N — SÜREKLİ
# ======================================================================

gerekli_n_surekli <- function(d, n_sabit, hedef_power, alfa, max_n) {

  maksimum_power <- power_surekli(n_karsi = max_n, n_sabit = n_sabit, d = d, alfa = alfa)

  if (maksimum_power < hedef_power) {
    return(
      data.frame(
        etki = d,
        gerekli_karsi_n = NA_integer_,
        toplam_n = NA_integer_,
        elde_edilen_power = maksimum_power,
        durum = paste0(
          n_sabit,
          " kişilik sabit grup ile %80 güce ",
          "yalnızca karşı grup artırılarak ulaşılamaz"
        )
      )
    )
  }

  alt <- 2L
  ust <- 2L

  while (
    power_surekli(n_karsi = ust, n_sabit = n_sabit, d = d, alfa = alfa) < hedef_power &&
    ust < max_n
  ) {
    ust <- min(ust * 2L, max_n)
  }

  while (alt < ust) {
    orta <- floor((alt + ust) / 2)
    pw <- power_surekli(n_karsi = orta, n_sabit = n_sabit, d = d, alfa = alfa)
    if (pw >= hedef_power) {
      ust <- orta
    } else {
      alt <- orta + 1L
    }
  }

  gerekli_n <- alt

  final_power <- power_surekli(n_karsi = gerekli_n, n_sabit = n_sabit, d = d, alfa = alfa)

  data.frame(
    etki = d,
    gerekli_karsi_n = gerekli_n,
    toplam_n = n_sabit + gerekli_n,
    elde_edilen_power = final_power,
    durum = "Hedef güce ulaşılıyor"
  )
}


# ======================================================================
# 17. TÜM d DEĞERLERİNİ HESAPLA
# ======================================================================

surekli_sonuclar <- bind_rows(
  lapply(
    etki_degerleri,
    gerekli_n_surekli,
    n_sabit = n_sabit,
    hedef_power = hedef_power,
    alfa = alfa,
    max_n = maksimum_karsi_grup
  )
)

surekli_sonuclar <- surekli_sonuclar %>%
  mutate(
    degisken_tipi = "Sürekli değişken",
    etki_olcusu = "Cohen's d",
    etki_yorumu = sapply(etki, etki_etiketi)
  )


# ======================================================================
#                  B. KATEGORİK DEĞİŞKENLER
# ======================================================================

# ======================================================================
# 18. KATEGORİK DEĞİŞKEN POWER FONKSİYONU
# ======================================================================

power_kategorik <- function(n_karsi, n_sabit, h, alfa) {
  sonuc <- pwr::pwr.2p2n.test(
    h = h,
    n1 = n_sabit,
    n2 = n_karsi,
    sig.level = alfa,
    alternative = "two.sided"
  )
  sonuc$power
}


# ======================================================================
# 19. GEREKLİ KARŞI GRUP N — KATEGORİK
# ======================================================================

gerekli_n_kategorik <- function(h, n_sabit, hedef_power, alfa, max_n) {

  maksimum_power <- power_kategorik(n_karsi = max_n, n_sabit = n_sabit, h = h, alfa = alfa)

  if (maksimum_power < hedef_power) {
    return(
      data.frame(
        etki = h,
        gerekli_karsi_n = NA_integer_,
        toplam_n = NA_integer_,
        elde_edilen_power = maksimum_power,
        durum = paste0(
          n_sabit,
          " kişilik sabit grup ile %80 güce ",
          "yalnızca karşı grup artırılarak ulaşılamaz"
        )
      )
    )
  }

  alt <- 2L
  ust <- 2L

  while (
    power_kategorik(n_karsi = ust, n_sabit = n_sabit, h = h, alfa = alfa) < hedef_power &&
    ust < max_n
  ) {
    ust <- min(ust * 2L, max_n)
  }

  while (alt < ust) {
    orta <- floor((alt + ust) / 2)
    pw <- power_kategorik(n_karsi = orta, n_sabit = n_sabit, h = h, alfa = alfa)
    if (pw >= hedef_power) {
      ust <- orta
    } else {
      alt <- orta + 1L
    }
  }

  gerekli_n <- alt

  final_power <- power_kategorik(n_karsi = gerekli_n, n_sabit = n_sabit, h = h, alfa = alfa)

  data.frame(
    etki = h,
    gerekli_karsi_n = gerekli_n,
    toplam_n = n_sabit + gerekli_n,
    elde_edilen_power = final_power,
    durum = "Hedef güce ulaşılıyor"
  )
}


# ======================================================================
# 20. TÜM h DEĞERLERİNİ HESAPLA
# ======================================================================

kategorik_sonuclar <- bind_rows(
  lapply(
    etki_degerleri,
    gerekli_n_kategorik,
    n_sabit = n_sabit,
    hedef_power = hedef_power,
    alfa = alfa,
    max_n = maksimum_karsi_grup
  )
)

kategorik_sonuclar <- kategorik_sonuclar %>%
  mutate(
    degisken_tipi = "Kategorik değişken",
    etki_olcusu = "Cohen's h",
    etki_yorumu = sapply(etki, etki_etiketi)
  )


# ======================================================================
# 21. EK 1 — NA (ULAŞILAMAZ) DURUMUNDA SABİT GRUBU 1'ER ARTIR
# ======================================================================
# Bir etki büyüklüğü için hedef güce ulaşılamıyorsa (NA), sabit grup
# sayısı 1'er artırılır. Hedef güce İLK ulaşılan sabit N ve o N'deki
# gerekli karşı grup N kaydedilir.
# Not: Ana n_sabit DEĞİŞMEZ. Bu yalnızca raporlama içindir.
# ======================================================================

ilk_ulasilan_nokta <- function(etki, n_baslangic, tip, hedef_power, alfa, max_n) {

  n_deneme <- n_baslangic

  repeat {
    n_deneme <- n_deneme + 1L
    if (tip == "surekli") {
      pw <- power_surekli(max_n, n_deneme, etki, alfa)
    } else {
      pw <- power_kategorik(max_n, n_deneme, etki, alfa)
    }
    if (pw >= hedef_power) break
    if (n_deneme > 1000000L) return(c(NA_integer_, NA_integer_))
  }

  if (tip == "surekli") {
    sonuc <- gerekli_n_surekli(etki, n_deneme, hedef_power, alfa, max_n)
  } else {
    sonuc <- gerekli_n_kategorik(etki, n_deneme, hedef_power, alfa, max_n)
  }

  c(n_deneme, sonuc$gerekli_karsi_n)
}

na_icin_ek_bilgi <- function(sonuclar, tip) {
  sonuclar$ilk_ulasilan_sabit_n <- NA_integer_
  sonuclar$ilk_ulasilan_karsi_n <- NA_integer_
  for (i in seq_len(nrow(sonuclar))) {
    if (is.na(sonuclar$gerekli_karsi_n[i])) {
      nokta <- ilk_ulasilan_nokta(
        etki = sonuclar$etki[i],
        n_baslangic = n_sabit,
        tip = tip,
        hedef_power = hedef_power,
        alfa = alfa,
        max_n = maksimum_karsi_grup
      )
      sonuclar$ilk_ulasilan_sabit_n[i] <- nokta[1]
      sonuclar$ilk_ulasilan_karsi_n[i] <- nokta[2]
    }
  }
  sonuclar
}

surekli_sonuclar   <- na_icin_ek_bilgi(surekli_sonuclar, "surekli")
kategorik_sonuclar <- na_icin_ek_bilgi(kategorik_sonuclar, "kategorik")


# ======================================================================
# 22. YENİ — ELDEKİ NEGATİF HASTA SAYISIYLA GÜÇ
# ======================================================================
# Pozitif N = n_sabit, negatif N = eldeki negatif hasta sayısı iken
# her etki büyüklüğü için ulaşılan güç ve eldeki sayının yeterli olup olmadığı.

mevcut_guc <- function(etki, tip) {
  if (n_karsi_mevcut < 2) return(NA_real_)
  if (tip == "surekli") {
    power_surekli(n_karsi_mevcut, n_sabit, etki, alfa)
  } else {
    power_kategorik(n_karsi_mevcut, n_sabit, etki, alfa)
  }
}

surekli_sonuclar <- surekli_sonuclar %>%
  mutate(
    mevcut_negatif_n = n_karsi_mevcut,
    mevcut_n_guc     = sapply(etki, mevcut_guc, tip = "surekli"),
    mevcut_n_yeterli = ifelse(!is.na(gerekli_karsi_n) & gerekli_karsi_n <= n_karsi_mevcut, "Evet", "Hayır")
  )

kategorik_sonuclar <- kategorik_sonuclar %>%
  mutate(
    mevcut_negatif_n = n_karsi_mevcut,
    mevcut_n_guc     = sapply(etki, mevcut_guc, tip = "kategorik"),
    mevcut_n_yeterli = ifelse(!is.na(gerekli_karsi_n) & gerekli_karsi_n <= n_karsi_mevcut, "Evet", "Hayır")
  )

# Eldeki iki grupla %80 güçle saptanabilecek en küçük etki büyüklüğü
mdes_d <- tryCatch(
  pwr::pwr.t2n.test(n1 = n_sabit, n2 = n_karsi_mevcut, sig.level = alfa,
                    power = hedef_power, alternative = "two.sided")$d,
  error = function(e) NA_real_
)
mdes_h <- tryCatch(
  pwr::pwr.2p2n.test(n1 = n_sabit, n2 = n_karsi_mevcut, sig.level = alfa,
                     power = hedef_power, alternative = "two.sided")$h,
  error = function(e) NA_real_
)

mdes_metni <- paste0(
  "Eldeki örneklemle (pozitif N = ", n_sabit, ", negatif N = ", n_karsi_mevcut,
  ") %", round(hedef_power * 100), " güç ve α = ", alfa,
  " ile saptanabilecek en küçük etki büyüklüğü: Cohen's d = ", sprintf("%.2f", mdes_d),
  ", Cohen's h = ", sprintf("%.2f", mdes_h), "."
)


# ======================================================================
# 23. SONUÇLARI BİRLEŞTİR
# ======================================================================

tum_sonuclar <- bind_rows(surekli_sonuclar, kategorik_sonuclar)


# ======================================================================
# 24. ANA YAYIN TABLOSU  (NA ise altta parantez içinde ilk ulaşılan nokta)
# ======================================================================

ana_tablo <- tum_sonuclar %>%
  transmute(
    `Değişken türü` = degisken_tipi,
    `Etki ölçüsü` = etki_olcusu,
    `Etki büyüklüğü` = sprintf("%.2f", etki),
    `Etki düzeyi` = etki_yorumu,
    `Sabit grup N (pozitif)` = n_sabit,
    `Gerekli negatif grup N` =
      ifelse(
        is.na(gerekli_karsi_n),
        ifelse(
          is.na(ilk_ulasilan_sabit_n),
          "Ulaşılamaz",
          paste0(
            "Ulaşılamaz\n(Sabit grup N = ", ilk_ulasilan_sabit_n,
            " olunca ilk ulaşılır; karşı grup N = ", ilk_ulasilan_karsi_n, ")"
          )
        ),
        as.character(gerekli_karsi_n)
      ),
    `Toplam N` = ifelse(is.na(toplam_n), "—", as.character(toplam_n)),
    `Güç` = sprintf("%.1f%%", elde_edilen_power * 100),
    `Sonuç` = durum,
    `Eldeki negatif N ile güç` = yuzde(mevcut_n_guc),
    `Eldeki negatif N yeterli mi?` = mevcut_n_yeterli
  )


# ======================================================================
# 25. KONSOLDA ANA SONUÇLARI GÖSTER
# ======================================================================

cat("\n============================================================\n")
cat("GEREKLİ NEGATİF GRUP ÖRNEKLEM SAYILARI\n")
cat("============================================================\n\n")
print(as.data.frame(ana_tablo), row.names = FALSE)


# ======================================================================
# 26. SÜREKLİ VE KATEGORİK TABLOLAR
# ======================================================================

surekli_tablo <- ana_tablo %>%
  filter(`Değişken türü` == "Sürekli değişken") %>%
  select(-`Değişken türü`)

kategorik_tablo <- ana_tablo %>%
  filter(`Değişken türü` == "Kategorik değişken") %>%
  select(-`Değişken türü`)


# ======================================================================
# 27. PRATİK ÖZET TABLO
# ======================================================================

pratik_gerekli <- tum_sonuclar %>%
  transmute(
    etki,
    sutun = if_else(degisken_tipi == "Sürekli değişken",
                    "Gerekli negatif N — sürekli (d)", "Gerekli negatif N — kategorik (h)"),
    gerekli_karsi_n
  ) %>%
  pivot_wider(names_from = sutun, values_from = gerekli_karsi_n)

pratik_guc <- tum_sonuclar %>%
  transmute(
    etki,
    sutun = if_else(degisken_tipi == "Sürekli değişken",
                    "Eldeki N ile güç — sürekli (d)", "Eldeki N ile güç — kategorik (h)"),
    guc = yuzde(mevcut_n_guc)
  ) %>%
  pivot_wider(names_from = sutun, values_from = guc)

pratik_tablo <- pratik_gerekli %>%
  left_join(pratik_guc, by = "etki") %>%
  mutate(`Etki düzeyi` = sapply(etki, etki_etiketi)) %>%
  select(`Etki büyüklüğü` = etki, `Etki düzeyi`, everything())

cat("\n============================================================\n")
cat("PRATİK ÖZET TABLO\n")
cat("============================================================\n\n")
print(as.data.frame(pratik_tablo), row.names = FALSE)

cat("\n", mdes_metni, "\n", sep = "")

## 5. Tablolar ve dosyaları Drive'a kaydetme

In [ ]:
%%R
# ======================================================================
# 28. GT — SÜREKLİ DEĞİŞKENLER
# ======================================================================

varsayim_notu <- paste0(
  "**Varsayımlar:** iki yönlü α = ", alfa, ", hedef güç = %", round(hedef_power * 100), ". ",
  "Sabit grup N = ", n_sabit, " (histerektomi + re-LEEP dosyalarındaki benzersiz cerrahi sınır pozitif hasta, ",
  "Excel'den otomatik). Eldeki cerrahi sınır negatif hasta sayısı = ", n_karsi_mevcut, ". "
)

gt_surekli <- surekli_tablo %>%
  mutate(`Gerekli negatif grup N` = gsub("\n", "<br>", `Gerekli negatif grup N`)) %>%
  gt() %>%
  fmt_markdown(columns = `Gerekli negatif grup N`) %>%
  tab_header(
    title = md(paste0("**", n_sabit, " kişilik cerrahi sınır pozitif gruba karşı gerekli negatif grup sayısı**")),
    subtitle = md("**Sürekli değişkenler — Cohen's d**")
  ) %>%
  cols_align(align = "center", columns = everything()) %>%
  tab_source_note(source_note = md(paste0(varsayim_notu, "Sürekli değişkenler için Cohen's d kullanılmıştır."))) %>%
  tab_options(
    table.font.size = px(13),
    heading.title.font.size = px(16),
    heading.subtitle.font.size = px(14),
    data_row.padding = px(7)
  )


# ======================================================================
# 29. GT — KATEGORİK DEĞİŞKENLER
# ======================================================================

gt_kategorik <- kategorik_tablo %>%
  mutate(`Gerekli negatif grup N` = gsub("\n", "<br>", `Gerekli negatif grup N`)) %>%
  gt() %>%
  fmt_markdown(columns = `Gerekli negatif grup N`) %>%
  tab_header(
    title = md(paste0("**", n_sabit, " kişilik cerrahi sınır pozitif gruba karşı gerekli negatif grup sayısı**")),
    subtitle = md("**İkili kategorik değişkenler — Cohen's h**")
  ) %>%
  cols_align(align = "center", columns = everything()) %>%
  tab_source_note(source_note = md(paste0(varsayim_notu, "İkili kategorik değişkenler için Cohen's h kullanılmıştır."))) %>%
  tab_options(
    table.font.size = px(13),
    heading.title.font.size = px(16),
    heading.subtitle.font.size = px(14),
    data_row.padding = px(7)
  )


# ======================================================================
# 30. GT — PRATİK ÖZET
# ======================================================================

gt_pratik <- pratik_tablo %>%
  gt() %>%
  sub_missing(missing_text = "Ulaşılamaz") %>%
  tab_header(
    title = md(paste0("**Cerrahi sınır pozitif (N = ", n_sabit, ") vs cerrahi sınır negatif (eldeki N = ",
                      n_karsi_mevcut, ")**")),
    subtitle = md(paste0("Gerekli negatif grup sayısı ve eldeki sayıyla güç | Hedef güç = %",
                         round(hedef_power * 100), " | α = ", alfa, " | İki yönlü"))
  ) %>%
  cols_align(align = "center", columns = everything()) %>%
  tab_source_note(source_note = md(paste0(
    "Sabit grup N = ", n_sabit, ", histerektomi + re-LEEP dosyalarındaki benzersiz kişi sayısından ",
    "otomatik hesaplanmıştır. Sürekli değişkenler için Cohen's d; ikili kategorik değişkenler için ",
    "Cohen's h kullanılmıştır."
  ))) %>%
  tab_source_note(source_note = mdes_metni) %>%
  tab_options(
    table.font.size = px(13),
    heading.title.font.size = px(16),
    heading.subtitle.font.size = px(13),
    data_row.padding = px(7)
  )


# ======================================================================
# 31. COLAB'DA GÖSTERİLECEK TABLOLAR (bir sonraki hücre gösterir)
# ======================================================================

html_parca <- function(gt_tablo) as.character(gt::as_raw_html(gt_tablo, inline_css = FALSE))

colab_html <- c(
  html_parca(gt_pratik),
  html_parca(gt_surekli),
  html_parca(gt_kategorik)
)


# ======================================================================
# 32. WORD İÇİN FLEXTABLE
# ======================================================================

ft_yap <- function(veri) {
  ft <- flextable(as.data.frame(veri))
  ft <- theme_booktabs(ft)
  ft <- bold(ft, part = "header")
  ft <- align(ft, align = "center", part = "all")
  ft <- fontsize(ft, size = 9, part = "all")
  ft <- autofit(ft)
  fit_to_width(ft, max_width = 9.5)
}

ft_surekli   <- ft_yap(surekli_tablo)
ft_kategorik <- ft_yap(kategorik_tablo)
ft_pratik    <- ft_yap(pratik_tablo %>% mutate(across(starts_with("Gerekli"), ~ ifelse(is.na(.x), "Ulaşılamaz", as.character(.x)))))


# ======================================================================
# 33. DOSYA ADI: BUGÜNÜN TARİHİ + VERSİYON
# ======================================================================

bugun <- format(Sys.time(), "%Y-%m-%d", tz = saat_dilimi)
dosya_etiketi <- versiyonlu_ad(ana_klasor, cikti_adi, bugun)

word_yolu  <- file.path(ana_klasor, paste0(dosya_etiketi, ".docx"))
excel_yolu <- file.path(ana_klasor, paste0(dosya_etiketi, ".xlsx"))
html_yolu  <- file.path(ana_klasor, paste0(dosya_etiketi, ".html"))


# ======================================================================
# 34. WORD DOSYASI (yatay sayfa)
# ======================================================================

doc <- read_docx()
doc <- body_set_default_section(doc, prop_section(page_size = page_size(orient = "landscape")))

doc <- body_add_par(doc, "Örneklem büyüklüğü ve güç analizi", style = "heading 1")

doc <- body_add_par(
  doc,
  paste0(
    "Cerrahi sınır pozitif grup (histerektomi + re-LEEP) sabit grup olarak alınmış ve büyüklüğü ",
    "Excel dosyalarındaki benzersiz kişi sayısından otomatik olarak N = ", n_sabit,
    " bulunmuştur. Eldeki cerrahi sınır negatif hasta sayısı N = ", n_karsi_mevcut, "'dir. ",
    "İki yönlü alfa düzeyi ", alfa, " ve hedef istatistiksel güç %", round(hedef_power * 100),
    " olarak kabul edilmiştir. Farklı etki büyüklüğü senaryolarında hedef güce ulaşmak için ",
    "gerekli negatif grup büyüklüğü ve eldeki negatif hasta sayısıyla ulaşılan güç hesaplanmıştır."
  )
)

doc <- body_add_par(doc, "Pratik özet", style = "heading 2")
doc <- body_add_flextable(doc, value = ft_pratik)
doc <- body_add_par(doc, mdes_metni)

doc <- body_add_par(doc, "Sürekli değişkenler (Cohen's d)", style = "heading 2")
doc <- body_add_flextable(doc, value = ft_surekli)

doc <- body_add_par(doc, "İkili kategorik değişkenler (Cohen's h)", style = "heading 2")
doc <- body_add_flextable(doc, value = ft_kategorik)

doc <- body_add_par(
  doc,
  paste0(
    "Sürekli değişkenlerde etki büyüklüğü Cohen's d, ikili kategorik değişkenlerde Cohen's h ile ",
    "ifade edilmiştir. Çok küçük etki büyüklüklerinde yalnızca karşı grubun artırılması hedeflenen ",
    "güce ulaşmak için yeterli olmayabilir."
  )
)

print(doc, target = word_yolu)


# ======================================================================
# 35. EXCEL DOSYASI
# ======================================================================

veri_ozeti <- data.frame(
  Degisken = c(
    "Analiz tarihi",
    "Grup tanımı",
    "Histerektomi dosyası benzersiz hasta",
    "Re-LEEP dosyası benzersiz hasta",
    "Negatif dosyası benzersiz hasta",
    "Sabit grup N (cerrahi sınır pozitif)",
    "Eldeki negatif grup N",
    "Alfa",
    "Hedef power",
    "Saptanabilir en küçük Cohen's d (eldeki N)",
    "Saptanabilir en küçük Cohen's h (eldeki N)"
  ),
  Deger = c(
    bugun,
    grup_tanimi,
    n_distinct(histerektomi$hasta_no),
    n_distinct(releep$hasta_no),
    n_distinct(negatif$hasta_no),
    n_sabit,
    n_karsi_mevcut,
    alfa,
    hedef_power,
    sprintf("%.3f", mdes_d),
    sprintf("%.3f", mdes_h)
  )
)

write_xlsx(
  list(
    "Veri_Ozeti"          = veri_ozeti,
    "Dosya_Ozeti"         = as.data.frame(dosya_ozeti),
    "Grup_Dagilimi"       = as.data.frame(grup_dagilimi),
    "Sinir_Kodlari"       = as.data.frame(sinir_kod_tablosu),
    "Pratik_Ozet"         = as.data.frame(pratik_tablo),
    "Surekli_Cohen_d"     = as.data.frame(surekli_sonuclar),
    "Kategorik_Cohen_h"   = as.data.frame(kategorik_sonuclar),
    "Tum_Sonuclar"        = as.data.frame(tum_sonuclar),
    "Tekrarlayan_Kayit"   = bos_ise_bilgi(tekrarlayan_kayitlar),
    "Grup_Disi_Kalanlar"  = bos_ise_bilgi(grup_disi_kalanlar),
    "Analizdeki_Hastalar" = as.data.frame(analiz %>% select(hasta_no, grup, kaynak, sinir_durumu, excel_satiri))
  ),
  path = excel_yolu
)


# ======================================================================
# 36. HTML DOSYASI (üç tablo tek sayfada)
# ======================================================================

html_icerik <- paste0(
  "<!DOCTYPE html>\n<html lang=\"tr\">\n<head>\n<meta charset=\"utf-8\">\n",
  "<title>", dosya_etiketi, "</title>\n</head>\n",
  "<body style=\"font-family: Arial, sans-serif; margin: 24px;\">\n",
  paste(colab_html, collapse = "\n<br><br>\n"),
  "\n</body>\n</html>\n"
)
writeLines(enc2utf8(html_icerik), html_yolu, useBytes = TRUE)


# ======================================================================
# 37. DOSYA KONTROLÜ
# ======================================================================

cat("\n============================================================\n")
cat("DOSYA KONTROLÜ\n")
cat("============================================================\n\n")
cat("Word oluşturuldu mu?:", file.exists(word_yolu), "\n")
cat("Excel oluşturuldu mu?:", file.exists(excel_yolu), "\n")
cat("HTML oluşturuldu mu?:", file.exists(html_yolu), "\n\n")
cat("WORD:\n", word_yolu, "\n\n")
cat("EXCEL:\n", excel_yolu, "\n\n")
cat("HTML:\n", html_yolu, "\n\n")

cat("============================================================\n")
cat("ANALİZ TAMAMLANDI —", dosya_etiketi, "\n")
cat("============================================================\n\n")
cat("Sabit grup N (cerrahi sınır pozitif) =", n_sabit, "\n")
cat("Eldeki negatif grup N =", n_karsi_mevcut, "\n")
cat(mdes_metni, "\n")

## 6. Tabloları göster

In [ ]:
# ======================================================================
# TABLOLARI COLAB'DA GÖSTER
# ======================================================================
from IPython.display import HTML, display
from rpy2 import robjects

for parca in robjects.r("colab_html"):
    display(HTML(str(parca)))